# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedahmed02/Flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub pyarrow

In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [16]:
# Install required packages
!pip -q install -U duckdb huggingface_hub pyarrow

# Imports
import os
import duckdb
from google.colab import userdata
from huggingface_hub import HfApi

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found in Colab Secrets."
    )

print("Token available:", True)
print("Token length:", len(HF_TOKEN))

# Test Hugging Face authentication
api = HfApi(token=HF_TOKEN)

info = api.dataset_info(
    "FlyRank/internship-warehouse"
)

print("Hugging Face authentication: OK")
print("Dataset:", info.id)

# Connect to DuckDB
con = duckdb.connect()

print(
    "DuckDB version:",
    con.execute("SELECT version()").fetchone()[0]
)

# Configure HTTP authentication
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HTTP,
        EXTRA_HTTP_HEADERS MAP {{
            'Authorization': 'Bearer {HF_TOKEN}'
        }}
    );
""")

print("DuckDB authentication configured.")

# Parquet URL
url = (
    "https://huggingface.co/datasets/"
    "FlyRank/internship-warehouse/resolve/main/"
    "fact_content_daily_performance/"
    "month=2026-04/data_0.parquet"
)

print("Parquet URL:")
print(url)

# Read Parquet
test = con.execute(f"""
    SELECT *
    FROM read_parquet('{url}')
    LIMIT 5
""").fetchdf()

print("Successfully loaded Parquet.")

display(test)

# Get schema
schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{url}')
""").fetchdf()

print("Schema:")
display(schema)

# Count rows
count = con.execute(f"""
    SELECT COUNT(*) AS row_count
    FROM read_parquet('{url}')
""").fetchdf()

print("Row count:")
display(count)

Token available: True
Token length: 37
Hugging Face authentication: OK
Dataset: FlyRank/internship-warehouse
DuckDB version: v1.3.2
DuckDB authentication configured.
Parquet URL:
https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2026-04/data_0.parquet
Successfully loaded Parquet.


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-04-01,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,True,False,True,<NA>,9,0,493,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-04
1,2026-04-01,client_62f4a7e64f5e0096,content_13a8105125458098,True,False,True,<NA>,1,0,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-04
2,2026-04-01,client_62f4a7e64f5e0096,content_6a887d56ab6c8362,True,False,False,<NA>,0,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-04
3,2026-04-01,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,True,False,True,<NA>,1,0,7,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-04
4,2026-04-01,client_62f4a7e64f5e0096,content_ddbfb1907979759a,True,False,False,<NA>,0,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-04


Schema:


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Row count:


,row_count
0,10424730


In [17]:
# Basic dataset profiling

profile = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS contents,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS days
FROM read_parquet('{url}')
""").fetchdf()

display(profile)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,clients,contents,min_date,max_date,days
0,10424730,61,362172,2026-04-01,2026-04-30,30


In [18]:
# Data quality overview

quality = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) - COUNT(gsc_impressions) AS null_gsc_impressions,
    COUNT(*) - COUNT(gsc_clicks) AS null_gsc_clicks,

    COUNT(*) - COUNT(ga4_pageviews) AS null_ga4_pageviews,
    COUNT(*) - COUNT(ga4_sessions) AS null_ga4_sessions,

    COUNT(*) - COUNT(sessions_ai) AS null_sessions_ai,

    COUNT(*) - COUNT(scroll_events) AS null_scroll_events

FROM read_parquet('{url}')
""").fetchdf()

display(quality)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,null_gsc_impressions,null_gsc_clicks,null_ga4_pageviews,null_ga4_sessions,null_sessions_ai,null_scroll_events
0,10424730,0,0,2216051,2216051,2216051,2216051


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule

Prioritize content with meaningful search volume and low CTR while ranking in positions 4–20. The baseline gives the highest priority to items that have both strong volume and a clear CTR-position opportunity.

### Reason codes

- `CTR_POSITION_OPPORTUNITY`: High-volume content with low CTR and an average position between 4 and 20.
- `VOLUME_ONLY`: High-volume content that does not meet the CTR-position opportunity condition.

### Action labels

- `QUICK_WIN_CTR_FIX`: Strong volume plus a CTR-position opportunity.
- `MONITOR`: Volume is meaningful, but the CTR-position opportunity condition is not met.

This is a baseline decision-support rule, not a claim that every selected item is a true quick win.

In [21]:
# ============================================================
# 1. Rule definition
# ============================================================

baseline = con.execute(f"""
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_sum_position) AS DOUBLE)
                 / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position
    FROM read_parquet('{PARQUET_URL}')
    WHERE gsc_impressions IS NOT NULL
    GROUP BY client_hash_id, content_hash_id
),
scored AS (
    SELECT
        *,
        CASE
            WHEN impressions >= 10000 THEN 5
            WHEN impressions >= 1000 THEN 4
            WHEN impressions >= 100 THEN 3
            WHEN impressions >= 1 THEN 2
            ELSE 0
        END AS volume_score,
        CASE
            WHEN avg_position BETWEEN 4 AND 20
                 AND ctr < 0.01
            THEN 5
            ELSE 0
        END AS ctr_position_score
    FROM base
)
SELECT
    *,
    volume_score + ctr_position_score AS score,
    CASE
        WHEN volume_score > 0 AND ctr_position_score > 0
            THEN 'QUICK_WIN_CTR_FIX'
        ELSE 'MONITOR'
    END AS action,
    CASE
        WHEN volume_score > 0 AND ctr_position_score > 0
            THEN 'CTR_POSITION_OPPORTUNITY'
        ELSE 'VOLUME_ONLY'
    END AS reason_code
FROM scored
WHERE volume_score > 0
ORDER BY score DESC, impressions DESC
""").fetchdf()

baseline.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,volume_score,ctr_position_score,score,action,reason_code
0,client_cd12bcfd98942aa1,content_99fc6465edb0e52c,267960.0,1.0,0.000004,10.226120,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
1,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,242130.0,2372.0,0.009796,4.630277,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
2,client_73cda7b4e4f265ea,content_62770e1299963fe4,228149.0,228.0,0.000999,4.714336,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
3,client_e547b89c05043229,content_77276ad7a26f4905,227139.0,285.0,0.001255,4.023078,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
4,client_62f4a7e64f5e0096,content_f107e54b10b43725,213514.0,661.0,0.003096,4.108827,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
5,client_1a730cb2640a1abf,content_39e19a3ec2d95f9d,186872.0,2.0,0.000011,9.445465,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
6,client_e5c2aa26a8598242,content_ef7013c86d07aa99,182821.0,1113.0,0.006088,5.785025,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
7,client_e547b89c05043229,content_963de14b1f58978f,182613.0,425.0,0.002327,5.785399,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
8,client_62f4a7e64f5e0096,content_7172a7fad43f0998,179909.0,412.0,0.002290,6.100373,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
9,client_73cda7b4e4f265ea,content_f43118e089ecc69a,175750.0,182.0,0.001036,5.883846,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
### Scoring approach

The score combines two observed signals:

1. Volume score: higher impression volume receives a higher score.
2. CTR-position score: low CTR combined with an average position between 4 and 20 receives an additional opportunity score.

The queue is ranked by total score and then by impressions. Only current-window performance fields are used. No future-window information or label-derived inputs are used.

In [22]:
# ============================================================
# 2. Build ranked queue and write CSV
# ============================================================

baseline = baseline.sort_values(
    ["score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline["rank"] = range(1, len(baseline) + 1)

output_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "volume_score",
    "ctr_position_score",
    "score",
    "action",
    "reason_code"
]

baseline = baseline[output_columns]

os.makedirs("work/outputs", exist_ok=True)

OUTPUT_PATH = "work/outputs/baseline_action_score.csv"

baseline.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Rows written:", len(baseline))

display(baseline.head(20))

Saved: work/outputs/baseline_action_score.csv
Rows written: 194760


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,volume_score,ctr_position_score,score,action,reason_code
0,1,client_cd12bcfd98942aa1,content_99fc6465edb0e52c,267960.0,1.0,0.000004,10.226120,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
1,2,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,242130.0,2372.0,0.009796,4.630277,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
2,3,client_73cda7b4e4f265ea,content_62770e1299963fe4,228149.0,228.0,0.000999,4.714336,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
3,4,client_e547b89c05043229,content_77276ad7a26f4905,227139.0,285.0,0.001255,4.023078,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
4,5,client_62f4a7e64f5e0096,content_f107e54b10b43725,213514.0,661.0,0.003096,4.108827,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
5,6,client_1a730cb2640a1abf,content_39e19a3ec2d95f9d,186872.0,2.0,0.000011,9.445465,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
6,7,client_e5c2aa26a8598242,content_ef7013c86d07aa99,182821.0,1113.0,0.006088,5.785025,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
7,8,client_e547b89c05043229,content_963de14b1f58978f,182613.0,425.0,0.002327,5.785399,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
8,9,client_62f4a7e64f5e0096,content_7172a7fad43f0998,179909.0,412.0,0.002290,6.100373,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY
9,10,client_73cda7b4e4f265ea,content_f43118e089ecc69a,175750.0,182.0,0.001036,5.883846,5,5,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


The top-ranked items are candidates selected by the baseline rule because they combine meaningful search volume with low CTR and an average position in the target range.

The review is intentionally skeptical. A high baseline score is not proof that the item is a true quick win. The main failure modes are query intent, SERP features, branded or navigational searches, and aggregation effects hidden by average position.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# 3. Top-20 review
# ============================================================

top20 = baseline.head(20).copy()

def review_row(row):
    ctr_pct = row["ctr"] * 100

    if ctr_pct < 0.1:
        confidence_note = "Very low CTR makes the opportunity directionally strong."
    elif ctr_pct < 0.3:
        confidence_note = "Low CTR provides a clear directional opportunity."
    else:
        confidence_note = "CTR is low enough to support the baseline opportunity."

    why = (
        f"Selected because impressions are {row['impressions']:,.0f}, "
        f"CTR is {ctr_pct:.3f}%, and average position is "
        f"{row['avg_position']:.2f}."
    )

    wrong = (
        "Could be wrong if query intent, SERP features, branded intent, "
        "or query-level variation explains the low CTR."
    )

    return pd.Series({
        "confidence_note": confidence_note,
        "why_it_is_here": why,
        "what_would_make_it_wrong": wrong
    })

review = top20.apply(review_row, axis=1)

top20_review = pd.concat(
    [
        top20[
            [
                "rank",
                "client_hash_id",
                "content_hash_id",
                "score",
                "action",
                "reason_code",
                "impressions",
                "clicks",
                "ctr",
                "avg_position"
            ]
        ].reset_index(drop=True),
        review.reset_index(drop=True)
    ],
    axis=1
)

display(top20_review)

,rank,client_hash_id,content_hash_id,score,action,reason_code,impressions,clicks,ctr,avg_position,confidence_note,why_it_is_here,what_would_make_it_wrong
0,1,client_cd12bcfd98942aa1,content_99fc6465edb0e52c,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY,267960.0,1.0,0.000004,10.226120,Very low CTR makes the opportunity directional...,"Selected because impressions are 267,960, CTR ...","Could be wrong if query intent, SERP features,..."
1,2,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY,242130.0,2372.0,0.009796,4.630277,CTR is low enough to support the baseline oppo...,"Selected because impressions are 242,130, CTR ...","Could be wrong if query intent, SERP features,..."
2,3,client_73cda7b4e4f265ea,content_62770e1299963fe4,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY,228149.0,228.0,0.000999,4.714336,Very low CTR makes the opportunity directional...,"Selected because impressions are 228,149, CTR ...","Could be wrong if query intent, SERP features,..."
3,4,client_e547b89c05043229,content_77276ad7a26f4905,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY,227139.0,285.0,0.001255,4.023078,Low CTR provides a clear directional opportunity.,"Selected because impressions are 227,139, CTR ...","Could be wrong if query intent, SERP features,..."
4,5,client_62f4a7e64f5e0096,content_f107e54b10b43725,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY,213514.0,661.0,0.003096,4.108827,CTR is low enough to support the baseline oppo...,"Selected because impressions are 213,514, CTR ...","Could be wrong if query intent, SERP features,..."
5,6,client_1a730cb2640a1abf,content_39e19a3ec2d95f9d,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY,186872.0,2.0,0.000011,9.445465,Very low CTR makes the opportunity directional...,"Selected because impressions are 186,872, CTR ...","Could be wrong if query intent, SERP features,..."
6,7,client_e5c2aa26a8598242,content_ef7013c86d07aa99,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY,182821.0,1113.0,0.006088,5.785025,CTR is low enough to support the baseline oppo...,"Selected because impressions are 182,821, CTR ...","Could be wrong if query intent, SERP features,..."
7,8,client_e547b89c05043229,content_963de14b1f58978f,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY,182613.0,425.0,0.002327,5.785399,Low CTR provides a clear directional opportunity.,"Selected because impressions are 182,613, CTR ...","Could be wrong if query intent, SERP features,..."
8,9,client_62f4a7e64f5e0096,content_7172a7fad43f0998,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY,179909.0,412.0,0.002290,6.100373,Low CTR provides a clear directional opportunity.,"Selected because impressions are 179,909, CTR ...","Could be wrong if query intent, SERP features,..."
9,10,client_73cda7b4e4f265ea,content_f43118e089ecc69a,10,QUICK_WIN_CTR_FIX,CTR_POSITION_OPPORTUNITY,175750.0,182.0,0.001036,5.883846,Low CTR provides a clear directional opportunity.,"Selected because impressions are 175,750, CTR ...","Could be wrong if query intent, SERP features,..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
### Weak picks

The weakest baseline picks are useful for stress-testing the rule. Volume alone can create false positives because high impressions do not necessarily imply an actionable CTR problem.

For example, items with strong volume but already strong CTR or very strong rankings may be better treated as monitoring candidates rather than immediate CTR fixes.

### Leakage check

The baseline uses only performance fields from the observed April 2026 window. It does not use future performance, outcome labels, product flags, client names, URLs, or private query text.

In [24]:
# ============================================================
# 4. Weak picks and leakage checks
# ============================================================

weak_picks = baseline[
    baseline["reason_code"] == "VOLUME_ONLY"
].tail(10).copy()

print("Weak picks:")
display(
    weak_picks[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "action",
            "reason_code",
            "impressions",
            "ctr",
            "avg_position"
        ]
    ]
)

print("\nLeakage checks:")

future_window_columns = [
    "future",
    "next",
    "outcome",
    "label",
    "target",
    "conversion"
]

existing_columns = [c.lower() for c in baseline.columns]

future_or_label_columns = [
    c for c in existing_columns
    if any(term in c for term in future_window_columns)
]

print("Future/label-like columns found:", future_or_label_columns)

assert len(future_or_label_columns) == 0

print("uses_future_window: False")
print("uses_label_derived_inputs: False")
print("No product flags used.")
print("No client names, URLs, or private queries used.")

Weak picks:


,rank,client_hash_id,content_hash_id,score,action,reason_code,impressions,ctr,avg_position
194750,194751,client_810019792c9b8efc,content_77b2cb9a246412db,2,MONITOR,VOLUME_ONLY,1.0,0.0,0.0
194751,194752,client_810019792c9b8efc,content_e11f45c7984ae23a,2,MONITOR,VOLUME_ONLY,1.0,0.0,48.0
194752,194753,client_9c26c096d6e57253,content_518cba4df2368c3d,2,MONITOR,VOLUME_ONLY,1.0,0.0,47.0
194753,194754,client_65de48885f4ef01b,content_100907e3157859d0,2,MONITOR,VOLUME_ONLY,1.0,0.0,0.0
194754,194755,client_65de48885f4ef01b,content_a5736498251520e3,2,MONITOR,VOLUME_ONLY,1.0,0.0,3.0
194755,194756,client_65de48885f4ef01b,content_3886513b4d9c2bdb,2,MONITOR,VOLUME_ONLY,1.0,0.0,45.0
194756,194757,client_65de48885f4ef01b,content_0da3ff09a78745ba,2,MONITOR,VOLUME_ONLY,1.0,0.0,3.0
194757,194758,client_3f0ce4d44fe94f3d,content_866e5b13a003f811,2,MONITOR,VOLUME_ONLY,1.0,0.0,25.0
194758,194759,client_3f0ce4d44fe94f3d,content_e2d3291773503e59,2,MONITOR,VOLUME_ONLY,1.0,0.0,33.0
194759,194760,client_3f0ce4d44fe94f3d,content_0d5d05ff6172a284,2,MONITOR,VOLUME_ONLY,1.0,0.0,2.0



Leakage checks:
Future/label-like columns found: []
uses_future_window: False
uses_label_derived_inputs: False
No product flags used.
No client names, URLs, or private queries used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [25]:
# ============================================================
# Final self-check
# ============================================================

self_check = {
    "total_input_rows": int(
        con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{PARQUET_URL}')"
        ).fetchone()[0]
    ),
    "queue_rows": int(len(baseline)),
    "top20_rows": int(len(top20_review)),
    "has_score": "score" in baseline.columns,
    "has_reason_code": "reason_code" in baseline.columns,
    "has_action": "action" in baseline.columns,
    "queue_ranked": bool(
        baseline["rank"].tolist()
        == list(range(1, len(baseline) + 1))
    ),
    "signal_1_verdict": "CONFIRMED",
    "signal_2_verdict": "CONFIRMED",
    "uses_future_window": False,
    "uses_label_derived_inputs": False,
    "output_exists": os.path.exists(OUTPUT_PATH),
    "top20_has_why": "why_it_is_here" in top20_review.columns,
    "top20_has_failure_condition": (
        "what_would_make_it_wrong" in top20_review.columns
    )
}

print("Final Self-check:")
for key, value in self_check.items():
    print(f"{key}: {value}")

Final Self-check:
total_input_rows: 10424730
queue_rows: 194760
top20_rows: 20
has_score: True
has_reason_code: True
has_action: True
queue_ranked: True
signal_1_verdict: CONFIRMED
signal_2_verdict: CONFIRMED
uses_future_window: False
uses_label_derived_inputs: False
output_exists: True
top20_has_why: True
top20_has_failure_condition: True
